# Matched subject swap on VALID
Attach COCO images and the v175 prompt cache. Run on CPU, then download the review ZIP and open index.html.

In [ ]:

from collections import Counter, defaultdict
from hashlib import file_digest, sha256
from html import escape
from pathlib import Path
from random import Random
from PIL import Image, ImageDraw
import json
import shutil

DATA = Path('/kaggle/input/datasets')
COCO = DATA/'vuthetam/mscoco-2014'
CACHE = DATA/'ducanh2403/csa-prompt-full-karpathy-v175'
OUT = Path('/kaggle/working/relation_subject_swap_valid')
CACHE_SHA = 'e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0'
SEED = 1511

def overlap(a,b):
    x = max(0, min(a[2],b[2])-max(a[0],b[0]))
    y = max(0, min(a[3],b[3])-max(a[1],b[1]))
    intersection = x*y
    area = (a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])
    return intersection/(area-intersection)

def candidates(row):
    objects = {r['instance_id']:r for r in row['objects']}
    relations = row['relations']
    linked = {(r['subject_instance'],r['object_instance'],r['predicate']) for r in relations}
    for slot, rel in enumerate(relations):
        subject, obj = (objects[rel[k]] for k in ('subject_instance','object_instance'))
        if subject['label']!='person' or obj['label']=='person' or min(subject['IoU'],obj['IoU'])<.50:
            continue
        for foil in objects.values():
            if foil['label']!='person' or foil['IoU']<.50 or foil['roi'] in (subject['roi'],obj['roi']):
                continue
            if overlap(subject['roi_bbox'],foil['roi_bbox'])>.10:
                continue
            if (foil['instance_id'],obj['instance_id'],rel['predicate']) in linked:
                continue
            yield dict(coco_id=row['coco_id'],filename=row['filename'],relation_slot=slot,
                       predicate=rel['predicate'],text=rel['text'],original=subject,foil=foil,object=obj)

assert overlap([0,0,.5,.5],[.5,.5,1,1])==0
assert overlap([0,0,1,1],[0,0,1,1])==1
files=list(CACHE.rglob('prompt_links.jsonl'))
if len(files)!=1:
    raise ValueError(f'Expected one prompt_links.jsonl under {CACHE}')
with files[0].open('rb') as stream:
    if file_digest(stream,'sha256').hexdigest()!=CACHE_SHA:
        raise ValueError('Expected the exact v175 prompt cache')
rng=Random(SEED)
by_predicate=defaultdict(list)
valid=set()
total_pairs=0
with files[0].open(encoding='utf-8') as stream:
    for line in stream:
        row=json.loads(line)
        if row['group']!='val':
            continue
        cid=row['coco_id']
        if cid in valid:
            raise ValueError(f'Duplicate VALID image {cid}')
        valid.add(cid)
        found=list(candidates(row))
        total_pairs+=len(found)
        if found:
            chosen=rng.choice(found)
            by_predicate[chosen['predicate']].append(chosen)
if len(valid)!=5000:
    raise ValueError('Expected 5000 VALID images')
counts={k:len(v) for k,v in sorted(by_predicate.items())}
for pool in by_predicate.values():
    rng.shuffle(pool)
sample=[]
while len(sample)<min(32,sum(counts.values())):
    for key in sorted(by_predicate):
        pool=by_predicate[key]
        if pool and len(sample)<32:
            sample.append(pool.pop())
if not sample:
    raise ValueError('No matched subject-swap candidates on VALID')

wanted={r['filename'] for r in sample}
paths={}
for path in COCO.rglob('COCO_*.jpg'):
    if path.name in wanted:
        if path.name in paths:
            raise ValueError(f'Duplicate image {path.name}')
        paths[path.name]=path
if paths.keys()!=wanted:
    raise FileNotFoundError(f'Missing COCO images: {sorted(wanted-paths.keys())}')
(OUT/'images').mkdir(parents=True,exist_ok=False)
options='<option value="">Chưa chấm</option><option value="yes">Đúng</option><option value="no">Sai</option><option value="unclear">Không rõ</option>'
cards=[]
for i,item in enumerate(sample,1):
    item['id']=i
    item['original_letter']=rng.choice(('A','B'))
    alternative='B' if item['original_letter']=='A' else 'A'
    path=paths[item['filename']]
    item['image_sha256']=sha256(path.read_bytes()).hexdigest()
    with Image.open(path) as original:
        image=original.convert('RGB')
    image.thumbnail((820,620))
    draw=ImageDraw.Draw(image)
    for name,tag,color in [('original',item['original_letter'],'#ffb531'),('foil',alternative,'#48f4fc'),('object','O','#8bff78')]:
        x,y,r,b=item[name]['roi_bbox']
        x,y,r,b=(x*(image.width-1),y*(image.height-1),r*(image.width-1),b*(image.height-1))
        draw.rectangle((x,y,r,b),outline=color,width=3)
        draw.text((x+3,y+3),tag,fill=color,stroke_width=1,stroke_fill='black')
    image.save(OUT/'images'/f'{i}.jpg',quality=90)
    cards.append(f'<section data-id="{i}"><h2>Ảnh {i} · COCO {item["coco_id"]}</h2>'
                 f'<img src="images/{i}.jpg" alt="Ảnh có hai người A, B và đối tượng O">'
                 f'<p><b>{escape(item["text"])}</b></p>'
                 f'<label>Ba box khoanh đúng hai người và vật? <select class="boxes">{options}</select></label>'
                 f'<label>Quan hệ giữa người A và vật O đúng? <select class="a">{options}</select></label>'
                 f'<label>Quan hệ giữa người B và vật O đúng? <select class="b">{options}</select></label></section>')
summary=dict(valid_images=len(valid),eligible_images=sum(counts.values()),candidate_swaps=total_pairs,
             eligible_by_predicate=counts,review_by_predicate=dict(Counter(r['predicate'] for r in sample)))
manifest=dict(split='val',cache_sha256=CACHE_SHA,seed=SEED,summary=summary,sample=sample)
raw=json.dumps(manifest,indent=2,ensure_ascii=False).encode('utf-8')
(OUT/'manifest.json').write_bytes(raw)
page='''<!doctype html><html lang="vi"><meta charset="utf-8"><title>Chấm liên kết prompt với vùng</title>
<style>body{max-width:900px;margin:30px auto;padding:0 16px;font:16px system-ui}section{border-top:1px solid #bbb;padding:14px 0}img{max-width:100%}label{display:block;margin:8px 0}</style>
<h1>Chấm liên kết prompt với vùng</h1><p>Đọc prompt rồi chấm riêng từng người với cùng vật O. Hai người có thể cùng đúng hoặc cùng sai. Chọn “Không rõ” khi thiếu bằng chứng.</p>
<button id="save">Tải nhãn JSON</button>'''+''.join(cards)+'''
<script>
document.getElementById('save').onclick=()=>{
 const records=[...document.querySelectorAll('section')].map(s=>Object.fromEntries([
   ['id',Number(s.dataset.id)],...['boxes','a','b'].map(k=>[k,s.querySelector('.'+k).value])
 ]));
 if(records.some(r=>!r.boxes||!r.a||!r.b)){alert('Còn mục chưa chấm.');return;}
 const link=document.createElement('a');
 link.href=URL.createObjectURL(new Blob([JSON.stringify({manifest_sha256:'MANIFEST_SHA',records},null,2)],{type:'application/json'}));
 link.download='relation_subject_swap_labels.json';link.click();
};
</script></html>'''
(OUT/'index.html').write_text(page.replace('MANIFEST_SHA',sha256(raw).hexdigest()),encoding='utf-8')
print(json.dumps(summary,indent=2))
print('SEND:',shutil.make_archive(str(OUT)+'_review','zip',OUT))
